### DATE:8/20/25

This is dedicated to training the bias factorized chromBPnet model across all 5 sets of randomized chromosomal folds (different spltis for valdiatio and training).
These individual models will be merged in subsequent steps to generate an averaged model for downstream use. 

In [2]:
import os

bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/bash"



# Submit with array index 0-4
!sbatch --array=0-4 $bash/train_model.sh

Submitted batch job 14782351


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def flatten_nested_dict(nested_dict: Dict, prefix: str = "") -> Dict:
    """
    Flatten a nested dictionary into a single-level dictionary.
    
    Parameters:
    -----------
    nested_dict : Dict
        Nested dictionary to flatten
    prefix : str
        Prefix to add to keys
        
    Returns:
    --------
    Dict
        Flattened dictionary
    """
    flattened = {}
    
    for key, value in nested_dict.items():
        new_key = f"{prefix}_{key}" if prefix else key
        
        if isinstance(value, dict):
            # Recursively flatten nested dictionaries
            flattened.update(flatten_nested_dict(value, new_key))
        else:
            # Add the value directly
            flattened[new_key] = value
    
    return flattened

def load_metrics_from_fold(fold_dir: Path) -> Dict:
    """
    Load metrics from a single fold directory.
    
    Parameters:
    -----------
    fold_dir : Path
        Path to the fold directory containing evaluation results
        
    Returns:
    --------
    Dict
        Dictionary containing flattened metrics from both bias_metrics.json and chrombpnet_metrics.json
    """
    metrics = {}
    
    # Load bias metrics
    bias_metrics_file = fold_dir / "evaluation" / "bias_metrics.json"
    if bias_metrics_file.exists():
        try:
            with open(bias_metrics_file, 'r') as f:
                bias_data = json.load(f)
            
            # Flatten the nested structure
            flattened_bias = flatten_nested_dict(bias_data, "bias")
            metrics.update(flattened_bias)
            print(f"✓ Loaded {len(flattened_bias)} bias metrics from {fold_dir.name}")
            print(f"  Bias metrics: {list(flattened_bias.keys())}")
            
        except Exception as e:
            print(f"✗ Failed to load bias metrics from {fold_dir.name}: {e}")
    else:
        print(f"✗ Bias metrics file not found in {fold_dir.name}")
    
    # Load ChromBPnet metrics
    chrombpnet_metrics_file = fold_dir / "evaluation" / "chrombpnet_metrics.json"
    if chrombpnet_metrics_file.exists():
        try:
            with open(chrombpnet_metrics_file, 'r') as f:
                chrombpnet_data = json.load(f)
            
            # Flatten the nested structure
            flattened_chrombpnet = flatten_nested_dict(chrombpnet_data, "chrombpnet")
            metrics.update(flattened_chrombpnet)
            print(f"✓ Loaded {len(flattened_chrombpnet)} ChromBPnet metrics from {fold_dir.name}")
            print(f"  ChromBPnet metrics: {list(flattened_chrombpnet.keys())}")
            
        except Exception as e:
            print(f"✗ Failed to load ChromBPnet metrics from {fold_dir.name}: {e}")
    else:
        print(f"✗ ChromBPnet metrics file not found in {fold_dir.name}")
    
    return metrics

def collect_all_metrics(base_dir: Path) -> pd.DataFrame:
    """
    Collect metrics from all folds and organize into a DataFrame.
    
    Parameters:
    -----------
    base_dir : Path
        Base directory containing fold subdirectories
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with fold information and all metrics
    """
    all_metrics = []
    
    # Find all fold directories
    fold_dirs = [d for d in base_dir.iterdir() if d.is_dir() and d.name.startswith('fold_')]
    fold_dirs.sort(key=lambda x: int(x.name.split('_')[1]))
    
    print(f"Found {len(fold_dirs)} fold directories: {[d.name for d in fold_dirs]}")
    
    for fold_dir in fold_dirs:
        fold_num = int(fold_dir.name.split('_')[1])
        metrics = load_metrics_from_fold(fold_dir)
        
        if metrics:
            metrics['fold'] = fold_num
            metrics['model_name'] = f"fold_{fold_num}"
            all_metrics.append(metrics)
            print(f"✓ Added metrics for {fold_dir.name}: {len(metrics)-2} metrics")
        else:
            print(f"✗ No metrics found for {fold_dir.name}")
    
    if not all_metrics:
        raise ValueError("No metrics were successfully loaded from any fold")
    
    df = pd.DataFrame(all_metrics)
    print(f"\nDataFrame created with shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Check data types
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'fold']
    print(f"Numeric columns (excluding 'fold'): {list(numeric_cols)}")
    
    return df

def create_summary_table(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    """
    Create summary statistics table for all metrics.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing metrics from all folds
    output_dir : Path
        Directory to save the summary table
        
    Returns:
    --------
    pd.DataFrame
        Summary statistics table
    """
    # Exclude non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'fold']
    
    if len(numeric_cols) == 0:
        print("WARNING: No numeric columns found for summary statistics")
        return pd.DataFrame()
    
    summary_stats = df[numeric_cols].describe().T
    summary_stats['cv_score'] = summary_stats['std'] / np.abs(summary_stats['mean'])  # Coefficient of variation
    
    # Round to 4 decimal places
    summary_stats = summary_stats.round(4)
    
    # Save summary table
    summary_file = output_dir / "metrics_summary_table.csv"
    summary_stats.to_csv(summary_file)
    logger.info(f"Summary table saved to {summary_file}")
    
    # Also save a formatted version
    formatted_file = output_dir / "metrics_summary_formatted.txt"
    with open(formatted_file, 'w') as f:
        f.write("ChromBPnet Cross-Validation Metrics Summary\n")
        f.write("=" * 50 + "\n\n")
        f.write(summary_stats.to_string())
        f.write(f"\n\nTotal folds analyzed: {len(df)}\n")
    
    logger.info(f"Formatted summary saved to {formatted_file}")
    
    return summary_stats

def create_correlation_barplot(df: pd.DataFrame, output_dir: Path):
    """
    Create bar plots for correlation metrics (Pearson and Spearman).
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing metrics from all folds
    output_dir : Path
        Directory to save the plots
    """
    # Look for correlation metrics
    correlation_patterns = ['pearsonr', 'spearmanr']
    correlation_cols = []
    
    for col in df.columns:
        if any(pattern in col.lower() for pattern in correlation_patterns):
            correlation_cols.append(col)
    
    if not correlation_cols:
        logger.warning("No correlation metrics found for plotting")
        return
    
    logger.info(f"Found correlation metrics: {correlation_cols}")
    
    # Create subplot for each correlation metric
    n_metrics = len(correlation_cols)
    fig, axes = plt.subplots(n_metrics, 1, figsize=(12, 6*n_metrics))
    
    if n_metrics == 1:
        axes = [axes]
    
    for idx, metric in enumerate(correlation_cols):
        ax = axes[idx]
        
        # Create bar plot
        x_labels = [f"fold_{i}" for i in df['fold']]
        values = df[metric].values
        
        # Color bars based on positive/negative values
        colors = ['darkblue' if v >= 0 else 'darkred' for v in values]
        
        bars = ax.bar(x_labels, values, color=colors, alpha=0.7)
        
        # Add value labels on bars
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.01),
                   f'{value:.3f}', ha='center', va='bottom' if height >= 0 else 'top',
                   fontweight='bold')
        
        ax.set_ylabel('Correlation Coefficient')
        ax.set_title(f'{metric.replace("_", " ").title()}')
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax.set_ylim(min(values) - 0.1, max(values) + 0.1)
        
        # Rotate x-axis labels if needed
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    correlation_plot_file = output_dir / "correlation_metrics_barplot.png"
    plt.savefig(correlation_plot_file, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Correlation bar plot saved to {correlation_plot_file}")

def create_comprehensive_metrics_plot(df: pd.DataFrame, output_dir: Path):
    """
    Create comprehensive metrics visualization.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing metrics from all folds
    output_dir : Path
        Directory to save the plots
    """
    # Get numeric columns (excluding fold)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'fold']
    
    if len(numeric_cols) == 0:
        logger.warning("No numeric metrics found for plotting")
        return
    
    # Create a grid of subplots
    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes.flatten()
    elif n_cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()
    
    for idx, metric in enumerate(numeric_cols):
        if idx >= len(axes):
            break
            
        ax = axes[idx]
        
        # Create bar plot
        x_labels = [f"fold_{i}" for i in df['fold']]
        values = df[metric].values
        
        # Use a color scheme
        colors = plt.cm.viridis(np.linspace(0, 1, len(values)))
        
        bars = ax.bar(x_labels, values, color=colors, alpha=0.8)
        
        # Add value labels on bars
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.3f}', ha='center', va='bottom',
                   fontsize=8, fontweight='bold')
        
        ax.set_ylabel('Value')
        ax.set_title(f'{metric.replace("_", " ").title()}', fontsize=10)
        
        # Rotate x-axis labels
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    
    # Hide unused subplots
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    comprehensive_plot_file = output_dir / "all_metrics_barplot.png"
    plt.savefig(comprehensive_plot_file, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Comprehensive metrics plot saved to {comprehensive_plot_file}")

def create_metrics_heatmap(df: pd.DataFrame, output_dir: Path):
    """
    Create a heatmap showing metrics across folds.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing metrics from all folds
    output_dir : Path
        Directory to save the heatmap
    """
    # Get numeric columns (excluding fold)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'fold']
    
    if len(numeric_cols) == 0:
        logger.warning("No numeric metrics found for heatmap")
        return
    
    # Prepare data for heatmap
    heatmap_data = df[numeric_cols].T
    heatmap_data.columns = [f"Fold {i}" for i in df['fold']]
    
    # Create heatmap
    plt.figure(figsize=(10, max(6, len(numeric_cols) * 0.5)))
    
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlBu_r', 
                center=0, cbar_kws={'label': 'Metric Value'})
    
    plt.title('ChromBPnet Metrics Across Folds')
    plt.ylabel('Metrics')
    plt.xlabel('Cross-Validation Folds')
    plt.tight_layout()
    
    heatmap_file = output_dir / "metrics_heatmap.png"
    plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Metrics heatmap saved to {heatmap_file}")

# Run the analysis
input_dir = Path("/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6/")
output_dir = Path("/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6/metrics_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

print("Starting ChromBPnet metrics analysis")
print(f"Input directory: {input_dir}")
print(f"Output directory: {output_dir}")

try:
    # Collect metrics from all folds
    df = collect_all_metrics(input_dir)
    
    # Display first few rows to verify the data
    print("\nFirst few rows of collected data:")
    print(df.head())
    
    # Save raw data
    raw_data_file = output_dir / "raw_metrics_data.csv"
    df.to_csv(raw_data_file, index=False)
    print(f"Raw metrics data saved to {raw_data_file}")
    
    # Create summary table
    summary_stats = create_summary_table(df, output_dir)
    if not summary_stats.empty:
        print("\nSummary Statistics:")
        print("=" * 50)
        print(summary_stats)
        
        # Create visualizations
        create_correlation_barplot(df, output_dir)
        create_comprehensive_metrics_plot(df, output_dir)
        create_metrics_heatmap(df, output_dir)
        
        # Print key insights
        print(f"\nAnalysis complete! Results saved to: {output_dir}")
        print(f"Total folds analyzed: {len(df)}")
        print(f"Metrics collected: {len(df.select_dtypes(include=[np.number]).columns) - 1}")
        
        # Identify best performing fold based on key metrics
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        numeric_cols = [col for col in numeric_cols if col != 'fold']
        
        # Look for correlation metrics (higher is better)
        pearson_cols = [col for col in numeric_cols if 'pearsonr' in col.lower()]
        if pearson_cols:
            for col in pearson_cols:
                best_fold = df.loc[df[col].idxmax(), 'fold']
                best_value = df.loc[df[col].idxmax(), col]
                print(f"Best performing fold for {col}: fold_{best_fold} (value: {best_value:.4f})")
        
        # Look for JSD metrics (lower is better)
        jsd_cols = [col for col in numeric_cols if 'jsd' in col.lower()]
        if jsd_cols:
            for col in jsd_cols:
                best_fold = df.loc[df[col].idxmin(), 'fold']
                best_value = df.loc[df[col].idxmin(), col]
                print(f"Best performing fold for {col}: fold_{best_fold} (value: {best_value:.4f})")
        
        # Look for MSE metrics (lower is better)
        mse_cols = [col for col in numeric_cols if 'mse' in col.lower()]
        if mse_cols:
            for col in mse_cols:
                best_fold = df.loc[df[col].idxmin(), 'fold']
                best_value = df.loc[df[col].idxmin(), col]
                print(f"Best performing fold for {col}: fold_{best_fold} (value: {best_value:.4f})")

except Exception as e:
    print(f"Analysis failed: {e}")
    import traceback
    traceback.print_exc()

2025-08-21 11:32:17,014 - INFO - Summary table saved to /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6/metrics_summary_table.csv
2025-08-21 11:32:17,026 - INFO - Formatted summary saved to /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6/metrics_summary_formatted.txt
2025-08-21 11:32:17,030 - INFO - Found correlation metrics: ['bias_counts_metrics_peaks_spearmanr', 'bias_counts_metrics_peaks_pearsonr', 'chrombpnet_counts_metrics_peaks_spearmanr', 'chrombpnet_counts_metrics_peaks_pearsonr']


Starting ChromBPnet metrics analysis
Input directory: /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6
Output directory: /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6
Found 5 fold directories: ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
✓ Loaded 5 bias metrics from fold_0
  Bias metrics: ['bias_counts_metrics_peaks_spearmanr', 'bias_counts_metrics_peaks_pearsonr', 'bias_counts_metrics_peaks_mse', 'bias_profile_metrics_peaks_median_jsd', 'bias_profile_metrics_peaks_median_norm_jsd']
✓ Loaded 5 ChromBPnet metrics from fold_0
  ChromBPnet metrics: ['chrombpnet_counts_metrics_peaks_spearmanr', 'chrombpnet_counts_metrics_peaks_pearsonr', 'chrombpnet_counts_metrics_peaks_mse', 'chrombpnet_profile_metrics_peaks_median_jsd', 'chrombpnet_profile_metrics_peaks_median_norm_jsd']
✓ Added metrics for fold_0: 10 metrics
✓ Loaded 5 bias metr

2025-08-21 11:32:18,872 - INFO - Correlation bar plot saved to /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6/correlation_metrics_barplot.png
2025-08-21 11:32:21,555 - INFO - Comprehensive metrics plot saved to /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6/all_metrics_barplot.png
2025-08-21 11:32:22,195 - INFO - Metrics heatmap saved to /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6/metrics_heatmap.png



Analysis complete! Results saved to: /gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/chromBPnet_training/outputs/pbmc-2_4_merged/chrombpnet_model_b0.6
Total folds analyzed: 5
Metrics collected: 10
Best performing fold for bias_counts_metrics_peaks_pearsonr: fold_3 (value: -0.4026)
Best performing fold for chrombpnet_counts_metrics_peaks_pearsonr: fold_4 (value: 0.8340)
Best performing fold for bias_profile_metrics_peaks_median_jsd: fold_4 (value: 0.5775)
Best performing fold for bias_profile_metrics_peaks_median_norm_jsd: fold_3 (value: 0.1882)
Best performing fold for chrombpnet_profile_metrics_peaks_median_jsd: fold_4 (value: 0.5598)
Best performing fold for chrombpnet_profile_metrics_peaks_median_norm_jsd: fold_3 (value: 0.2045)
Best performing fold for bias_counts_metrics_peaks_mse: fold_3 (value: 5.0810)
Best performing fold for chrombpnet_counts_metrics_peaks_mse: fold_1 (value: 0.3286)
